# 01 - Explore `products_cleaned.csv`

Use this notebook first. It profiles the raw EnterKomputer scrape before any cleaning.

Goals:

- Confirm the CSV schema.
- Inspect category and subcategory distribution.
- Identify rows that are useful for the PC Builder component catalog.
- Export lightweight profiling reports for review.

This notebook does **not** overwrite runtime files.

## How to use in Colab

1. Upload `products_cleaned.csv` when prompted, or place it beside this notebook.
2. Run all cells.
3. Review the generated reports under `data/cleaning_reports/`.

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
from IPython.display import display

CSV_PATH = Path('products_cleaned.csv')
REPORT_DIR = Path('data/cleaning_reports')
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if not CSV_PATH.exists():
    try:
        from google.colab import files
        print('Upload products_cleaned.csv')
        uploaded = files.upload()
        if uploaded:
            CSV_PATH = Path(next(iter(uploaded.keys())))
    except Exception as exc:
        raise FileNotFoundError('products_cleaned.csv was not found. Upload it or place it beside the notebook.') from exc

print('Using:', CSV_PATH)

In [ ]:
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]

expected = ['sku', 'name', 'category', 'subcategory', 'price_idr', 'stock_status', 'description', 'specifications', 'image_url', 'product_url', 'scraped_at']
missing = [c for c in expected if c not in df.columns]
extra = [c for c in df.columns if c not in expected]

print('Rows:', len(df))
print('Columns:', list(df.columns))
print('Missing expected columns:', missing)
print('Extra columns:', extra)
display(df.head(10))

In [ ]:
# Basic normalization for profiling only.
profile = df.copy()
profile['sku'] = profile['sku'].astype(str).str.strip()
profile['name'] = profile['name'].fillna('').astype(str).str.strip()
profile['category'] = profile['category'].fillna('').astype(str).str.strip()
profile['subcategory'] = profile['subcategory'].fillna('').astype(str).str.strip()
profile['price_idr'] = pd.to_numeric(profile['price_idr'], errors='coerce').fillna(0).astype('int64')
profile['stock_status'] = profile['stock_status'].fillna('').astype(str).str.strip()

summary = {
    'rows': int(len(profile)),
    'columns': list(profile.columns),
    'duplicate_sku_rows': int(profile.duplicated('sku').sum()),
    'zero_or_missing_price_rows': int((profile['price_idr'] <= 0).sum()),
    'missing_name_rows': int((profile['name'] == '').sum()),
    'in_stock_rows': int((profile['stock_status'].str.lower() == 'in_stock').sum()),
}
summary

In [ ]:
category_counts = (
    profile.groupby('category', dropna=False)
    .size()
    .reset_index(name='rows')
    .sort_values('rows', ascending=False)
)

display(category_counts.head(40))
category_counts.to_csv(REPORT_DIR / 'category_counts.csv', index=False)

In [ ]:
subcategory_counts = (
    profile.groupby(['category', 'subcategory'], dropna=False)
    .size()
    .reset_index(name='rows')
    .sort_values(['category', 'rows'], ascending=[True, False])
)

display(subcategory_counts.head(120))
subcategory_counts.to_csv(REPORT_DIR / 'subcategory_counts.csv', index=False)

In [ ]:
PC_BUILDER_SOURCE_CATEGORIES = {
    'Processor',
    'Motherboard',
    'VGA',
    'SSD',
    'Hard Drive',
    'PSU',
    'Cooler',
    'Casing',
    'LCD',
    'UPS',
}

candidate_profile = profile[profile['category'].isin(PC_BUILDER_SOURCE_CATEGORIES)].copy()

print('Raw rows:', len(profile))
print('PC Builder source candidate rows:', len(candidate_profile))
display(candidate_profile.groupby('category').size().reset_index(name='rows').sort_values('rows', ascending=False))
display(candidate_profile[['sku', 'name', 'category', 'subcategory', 'price_idr', 'stock_status', 'product_url']].head(50))

In [ ]:
# Rows intentionally excluded from PC Builder runtime data.
excluded = profile[~profile['category'].isin(PC_BUILDER_SOURCE_CATEGORIES)].copy()
excluded_counts = excluded.groupby('category').size().reset_index(name='rows').sort_values('rows', ascending=False)

display(excluded_counts.head(40))
excluded_counts.to_csv(REPORT_DIR / 'excluded_category_counts.csv', index=False)

In [ ]:
# Save a compact JSON report that can be committed or shared with collaborators.
report = {
    **summary,
    'category_counts': category_counts.head(100).to_dict(orient='records'),
    'pc_builder_candidate_rows': int(len(candidate_profile)),
    'pc_builder_candidate_categories': candidate_profile.groupby('category').size().sort_values(ascending=False).to_dict(),
}

(REPORT_DIR / 'raw_csv_profile.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
candidate_profile[['sku', 'name', 'category', 'subcategory', 'price_idr', 'stock_status', 'product_url']].head(250).to_csv(
    REPORT_DIR / 'component_candidate_preview.csv', index=False
)

print('Wrote reports to', REPORT_DIR)